# OneIE BKEE — tiếp tục từ checkpoint
Nạp output lượt chạy trước. Checkpoint legacy chỉ phục hồi trọng số; `last.mdl` mới phục hồi optimizer, scheduler và RNG tại cuối epoch. Không đặt timeout qua API.


In [ ]:
import socket
import subprocess
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
try:
    result = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, timeout=15)
    print(result.stdout or result.stderr)
except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
    print('GPU diagnostic:', type(exc).__name__)
problems = []
if not torch.cuda.is_available():
    problems.append('Phiên hiện tại không có CUDA. Kiểm tra Accelerator/GPU và trạng thái xác minh tài khoản trong Kaggle Settings.')
for host in ['pypi.org', 'github.com', 'huggingface.co']:
    try:
        with socket.create_connection((host, 443), timeout=10):
            print(host, 'OK')
    except OSError:
        problems.append(f'Không kết nối được {host}:443. Kiểm tra Internet trong Session options.')
if problems:
    raise RuntimeError('Preflight thất bại; chưa cài thư viện hoặc train:\n' + '\n'.join(problems))


In [3]:
%pip install -q "transformers==4.44.2" "tokenizers==0.19.1" "numpy<2" tqdm nltk lxml


Note: you may need to restart the kernel to use updated packages.


In [4]:
from pathlib import Path
import os, sys, json, subprocess, shutil
import torch
print("Python:", sys.version, "Torch:", torch.__version__)
assert torch.cuda.is_available(), "Bật GPU trong Kaggle Notebook settings"
print(torch.cuda.get_device_name(0))
# Tự tìm Dataset đã gắn vào notebook.
candidates = sorted({p.parent.parent for p in Path('/kaggle/input').rglob('prepare_oneie.py') if p.parent.name == 'scripts'})
assert len(candidates) == 1, f'Cần đúng một Dataset OneIE, tìm thấy: {candidates}'
PROJECT = candidates[0]
DATA = PROJECT / 'data/raw/BKEE'
WORK = Path('/kaggle/working/oneie-bkee')
ONEIE = WORK / 'OneIE'
PREPARED = WORK / 'data'
SMOKE = False  # Kiểm tra tích hợp trước.
AUTO_FULL = False  # Tự train đầy đủ sau khi smoke thành công.
SEED = 42
MAX_LENGTH = 512
MODEL = 'bert-base-multilingual-cased'
assert (PROJECT / 'scripts/prepare_oneie.py').exists(), PROJECT
assert all((DATA / f'{s}.json').exists() for s in ['train', 'dev', 'test']), DATA
WORK.mkdir(parents=True, exist_ok=True)
def run(*args, cwd=None):
    subprocess.run([str(x) for x in args], cwd=cwd, check=True)

# Attach the previous notebook output. Prefer a complete last.mdl for future runs.
resume_candidates = sorted(p for p in Path('/kaggle/input').rglob('last.mdl') if p.parent.parent.name == 'runs')
if not resume_candidates:
    resume_candidates = sorted(p for p in Path('/kaggle/input').rglob('best.role.mdl') if p.parent.parent.name == 'runs')
assert len(resume_candidates) == 1, f'Expected exactly one training checkpoint: {resume_candidates}'
RESUME = resume_candidates[0]
print('Checkpoint source:', RESUME)


ModuleNotFoundError: No module named 'torch'

In [ ]:
COMMIT = '3560bfc68cc8cf66ea8e226a614a1a75a9a5d979'
if not ONEIE.exists():
    run('git', 'clone', 'https://github.com/GerlinGreen/OneIE.git', ONEIE)
run('git', 'checkout', COMMIT, cwd=ONEIE)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ONEIE, text=True).strip() == COMMIT
run(sys.executable, PROJECT / 'scripts/patch_oneie.py', ONEIE)
run(sys.executable, PROJECT / 'scripts/prepare_oneie.py', '--input', DATA,
    '--output', PREPARED, '--model', MODEL, '--max-length', MAX_LENGTH)


In [ ]:
# Helpers embedded so the existing input Dataset need not be replaced.
(WORK / 'patch_oneie_resume.py').write_text('"""Add epoch-boundary resume after applying patch_oneie.py."""\nfrom pathlib import Path\nimport argparse\nimport shutil\n\n\ndef patch(root):\n    path = root / \'train.py\'\n    text = path.read_text()\n    shutil.copy2(Path(__file__).with_name(\'oneie_checkpoint.py\'), root / \'oneie_checkpoint.py\')\n    if \'# OneIE resume support\' in text:\n        return\n    edits = [\n        (\'# datasets\', \'\'\'# OneIE resume support\nfrom oneie_checkpoint import load_checkpoint, restore, save_checkpoint, fingerprint\nresume = load_checkpoint(config)\ndata_hashes = fingerprint(config)\n\n# datasets\'\'\'),\n        (\'vocabs = generate_vocabs([train_set, dev_set, test_set])\',\n         "vocabs = resume[\'vocabs\'] if resume else generate_vocabs([train_set, dev_set, test_set])"),\n        (\'valid_patterns = load_valid_patterns(config.valid_pattern_path, vocabs)\',\n         "valid_patterns = resume[\'valid\'] if resume else load_valid_patterns(config.valid_pattern_path, vocabs)"),\n        (\'for epoch in range(config.max_epoch):\', \'\'\'start_epoch, global_step, best_dev = restore(\n    resume, config, model, optimizer, schedule, output_dir, tasks)\ndel resume\nfor epoch in range(start_epoch, config.max_epoch):\'\'\'),\n        (\'torch.save(state, best_role_model)\', \'\'\'torch.save(dict(model=model.state_dict(), config=config.to_dict(),\n                                vocabs=vocabs, valid=valid_patterns), best_role_model)\'\'\'),\n        ("    print(\'Log file\', log_file)", \'\'\'    print(\'Log file\', log_file)\n    save_checkpoint(os.path.join(output_dir, \'last.mdl\'), model, config, vocabs,\n                    valid_patterns, optimizer, schedule, epoch, global_step,\n                    best_dev, data_hashes)\'\'\'),\n    ]\n    for old, new in edits:\n        if text.count(old) != 1:\n            raise ValueError(f\'Unexpected upstream source: {old}\')\n        text = text.replace(old, new)\n    compile(text, str(path), \'exec\')\n    path.write_text(text)\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(__doc__)\n    parser.add_argument(\'checkout\', type=Path)\n    patch(parser.parse_args().checkout)\n')
(WORK / 'oneie_checkpoint.py').write_text('"""Epoch-boundary checkpoint support for the pinned OneIE training script."""\nimport hashlib\nimport json\nimport os\nimport random\nimport shutil\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\n\ndef fingerprint(config):\n    return {\n        split: hashlib.sha256(Path(getattr(config, split + \'_file\')).read_bytes()).hexdigest()\n        for split in (\'train\', \'dev\', \'test\')\n    }\n\n\ndef load_checkpoint(config):\n    path = getattr(config, \'resume_checkpoint\', None)\n    if not path:\n        return None\n    # Only load the user\'s own trusted training artifacts (contains Python RNG state).\n    checkpoint = torch.load(path, map_location=\'cpu\', weights_only=False)\n    for key in (\'model\', \'vocabs\', \'valid\', \'config\'):\n        if key not in checkpoint:\n            raise ValueError(f\'Checkpoint missing {key}\')\n    for key, value in checkpoint[\'config\'].items():\n        allowed = {\'train_file\', \'dev_file\', \'test_file\', \'bert_cache_dir\', \'log_path\',\n                   \'valid_pattern_path\', \'use_gpu\', \'gpu_device\', \'resume_checkpoint\',\n                   \'resume_start_epoch\', \'resume_mode\'}\n        if key not in allowed and hasattr(config, key) and getattr(config, key) != value:\n            raise ValueError(f\'Resume config differs: {key}\')\n    if \'data_hashes\' in checkpoint and checkpoint[\'data_hashes\'] != fingerprint(config):\n        raise ValueError(\'Resume data differs from checkpoint\')\n    return checkpoint\n\n\ndef restore(checkpoint, config, model, optimizer, schedule, output_dir, tasks):\n    best = {key: -1 for key in tasks}\n    if checkpoint is None:\n        return 0, 0, best\n    model.load_state_dict(checkpoint[\'model\'], strict=True)\n    source = Path(config.resume_checkpoint).parent\n    history = source / \'log.txt\'\n    records = [json.loads(line) for line in history.read_text().splitlines()[1:]] if history.exists() else []\n    if checkpoint.get(\'checkpoint_version\') == 1:\n        start = checkpoint[\'epoch\'] + 1\n        optimizer.load_state_dict(checkpoint[\'optimizer\'])\n        schedule.load_state_dict(checkpoint[\'scheduler\'])\n        random.setstate(checkpoint[\'rng\'][\'python\'])\n        np.random.set_state(checkpoint[\'rng\'][\'numpy\'])\n        torch.set_rng_state(checkpoint[\'rng\'][\'torch\'])\n        if torch.cuda.is_available():\n            torch.cuda.set_rng_state_all(checkpoint[\'rng\'][\'cuda\'])\n        best = checkpoint[\'best_dev\']\n        step = checkpoint[\'global_step\']\n        mode = \'full-state, epoch boundary\'\n    else:\n        if not records:\n            raise ValueError(\'Legacy best.role.mdl requires its original log.txt\')\n        chosen = max(records, key=lambda row: row[\'dev\'][\'role\'][\'f\'])\n        start = chosen[\'epoch\'] + 1\n        step = 0\n        mode = \'weights-only; optimizer and scheduler restarted\'\n        print(\'WARNING:\', mode, flush=True)\n    if start >= config.max_epoch:\n        raise ValueError(\'Checkpoint already reached max_epoch\')\n    records = [row for row in records if row[\'epoch\'] < start]\n    if checkpoint.get(\'checkpoint_version\') != 1:\n        best = {key: max(row[\'dev\'][key][\'f\'] for row in records) for key in tasks}\n    with open(Path(output_dir) / \'log.txt\', \'a\') as stream:\n        for row in records:\n            stream.write(json.dumps(row) + \'\\n\')\n    for name in (\'best.role.mdl\', \'result.dev.json\', \'result.test.json\'):\n        if not (source / name).exists():\n            raise ValueError(f\'Resume artifact missing: {source / name}\')\n        shutil.copy2(source / name, Path(output_dir) / name)\n    (Path(output_dir) / \'resume.json\').write_text(json.dumps({\n        \'source\': str(config.resume_checkpoint), \'mode\': mode, \'start_epoch\': start,\n    }, indent=2))\n    print(f\'Resuming epoch {start} ({mode})\', flush=True)\n    return start, step, best\n\n\ndef save_checkpoint(path, model, config, vocabs, valid_patterns, optimizer,\n                    schedule, epoch, global_step, best_dev, data_hashes):\n    state = {\n        \'checkpoint_version\': 1, \'model\': model.state_dict(), \'config\': config.to_dict(),\n        \'vocabs\': vocabs, \'valid\': valid_patterns, \'optimizer\': optimizer.state_dict(),\n        \'scheduler\': schedule.state_dict(), \'epoch\': epoch, \'global_step\': global_step,\n        \'best_dev\': best_dev, \'data_hashes\': data_hashes,\n        \'rng\': {\'python\': random.getstate(), \'numpy\': np.random.get_state(),\n                \'torch\': torch.get_rng_state(),\n                \'cuda\': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []},\n    }\n    temporary = str(path) + \'.tmp\'\n    torch.save(state, temporary)\n    os.replace(temporary, path)\n')
run(sys.executable, WORK / 'patch_oneie_resume.py', ONEIE)


In [ ]:
TRAIN_DATA = PREPARED
if SMOKE:
    TRAIN_DATA = WORK / 'smoke-data'
    TRAIN_DATA.mkdir(exist_ok=True)
    # Chọn mẫu có event để kiểm tra cả loss argument; không dùng điểm smoke để báo cáo.
    for split, count in [('train', 32), ('dev', 8), ('test', 8)]:
        rows = [json.loads(line) for line in (PREPARED / f'{split}.jsonl').read_text().splitlines()]
        rows = sorted(rows, key=lambda row: not bool(row['event_mentions']))[:count]
        (TRAIN_DATA / f'{split}.jsonl').write_text(''.join(json.dumps(r, ensure_ascii=False)+'\n' for r in rows))
config = json.loads((ONEIE / 'config/example.json').read_text())
config.update({
    'bert_model_name': MODEL, 'bert_cache_dir': str(WORK / 'cache'),
    'train_file': str(TRAIN_DATA / 'train.jsonl'),
    'dev_file': str(TRAIN_DATA / 'dev.jsonl'),
    'test_file': str(TRAIN_DATA / 'test.jsonl'),
    'valid_pattern_path': str(PREPARED / 'patterns'),
    'log_path': str(WORK / ('smoke-runs' if SMOKE else 'runs')),
    'batch_size': 8, 'accumulate_step': 8, 'eval_batch_size': 1,
    'max_epoch': 1 if SMOKE else 30, 'warmup_epoch': 0 if SMOKE else 3,
    'max_length': MAX_LENGTH, 'seed': SEED,
    'use_gpu': True, 'gpu_device': 0,
    'symmetric_relations': [], 'ignore_title': False, 'ignore_first_header': False,
    'use_global_features': True,
})
Path(config['log_path']).mkdir(parents=True, exist_ok=True)
CONFIG = WORK / ('smoke-config.json' if SMOKE else 'config.json')
config['resume_checkpoint'] = str(RESUME)
CONFIG.write_text(json.dumps(config, indent=2))
print(CONFIG.read_text())


In [ ]:
# PYTHONHASHSEED cố định thứ tự các set trong upstream trước khi Python khởi động.
env = dict(os.environ, PYTHONHASHSEED=str(SEED), TOKENIZERS_PARALLELISM='false')
subprocess.run([sys.executable, '-u', 'train.py', '-c', str(CONFIG)],
               cwd=ONEIE, env=env, check=True)

if SMOKE and AUTO_FULL:
    print('Smoke thành công. Bắt đầu train toàn bộ BKEE.', flush=True)
    config.update({
        'train_file': str(PREPARED / 'train.jsonl'),
        'dev_file': str(PREPARED / 'dev.jsonl'),
        'test_file': str(PREPARED / 'test.jsonl'),
        'log_path': str(WORK / 'runs'),
        'max_epoch': 30, 'warmup_epoch': 3,
    })
    Path(config['log_path']).mkdir(parents=True, exist_ok=True)
    CONFIG = WORK / 'config.json'
    CONFIG.write_text(json.dumps(config, indent=2))
    SMOKE = False
    subprocess.run([sys.executable, '-u', 'train.py', '-c', str(CONFIG)],
                   cwd=ONEIE, env=env, check=True)


In [ ]:
logs = sorted(Path(config['log_path']).glob('*/log.txt'))
assert logs, 'Không tìm thấy log huấn luyện'
latest = logs[-1]
epochs = [json.loads(line) for line in latest.read_text().splitlines()[1:]]
best = max(epochs, key=lambda row: row['dev']['role']['f'])
print('Chọn epoch theo dev Role-C F1:', best['epoch'])
print(json.dumps(best, indent=2))
print('Checkpoint và prediction:', latest.parent)
# Download artifacts directly from Output; avoid duplicating multi-GB checkpoints.
